# FABLE Scenathon Trade Adjustment — Production Backend

This notebook is the **clean production reference** for the trade-adjustment backend.

It intentionally does **not** read or write Excel files. The Excel Office.js add-in is responsible for:

1. **Share your SCENATHON report** → read the minimal `SCENATHON_report` fields and send JSON.
2. **Inject adjusted trade** → retrieve adjusted rows, update `FullDataTradeAdj`, set `PathwaysSelection[S12] = Yes`, recalculate, and verify `FinalTradeAdj`.

The Python backend only handles:

- submission validation;
- deterministic report hashing;
- global trade adjustment across submitted countries/pathways;
- QA/blocker detection;
- generation of the minimal injection payload.

This separation removes the local Windows paths, workbook parsers, COM automation, one-off R/reference diagnostics, and batch workbook QA that were needed only during methodology validation.

In [ ]:
from pprint import pprint

from trade_engine import (
    ENGINE_VERSION,
    EXCLUDED_PRODUCTS,
    HISTORICAL_YEARS,
    PATHWAYS,
    TradeAdjustmentBlocked,
    build_injection_payload,
    calculate_trade_adjustment,
    normalize_submission,
)

print("Trade engine version:", ENGINE_VERSION)
print("Pathways:", PATHWAYS)
print("Historical years:", sorted(HISTORICAL_YEARS))
print("Excluded canonical products:", len(EXCLUDED_PRODUCTS))

## 1. Add-in submission contract

The add-in sends only the minimal report payload. A full `.xlsx` upload is not required.

Example:

```json
{
  "username": "demo_user",
  "country_id": "ARG",
  "pathway": "CurrentTrends",
  "rows": [
    {
      "Location": "Argentina",
      "Product": "barley",
      "Year": 2020,
      "Import_quantity": 10.0,
      "Export_quantity": 12.0
    }
  ]
}
```

`country_id` is optional. If it is not supplied, `Location` is used as the stable backend country key. This deliberately supports non-ISO regional identifiers such as `RMECAS`.

The server layer—not the calculation engine—should create `run_id`, iteration identifiers, timestamps, authentication, and persistence.

In [ ]:
sample_submission = {
    "username": "demo_user",
    "country_id": "ARG",
    "pathway": "CurrentTrends",
    "rows": [
        {
            "Location": "Argentina",
            "Product": "Barley",
            "Year": 2020,
            "Import_quantity": 10,
            "Export_quantity": 12,
        },
        {
            "Location": "Argentina",
            "Product": "Barley",
            "Year": 2025,
            "Import_quantity": 11,
            "Export_quantity": 13,
        },
    ],
}

normalized_submission = normalize_submission(sample_submission)

print("Validated submission")
print("Country:", normalized_submission["country_id"])
print("Pathway:", normalized_submission["pathway"])
print("Rows:", normalized_submission["row_count"])
print("Report hash:", normalized_submission["report_hash"])

## 2. Production methodology

The engine uses the validated production rules:

- product matching key = `Product.strip().casefold()`;
- no automatic semantic mapping such as `lentil ↔ lentils`;
- global totals are calculated by **pathway × canonical product × year**;
- 2020 global import/export imbalance defines `alpha`;
- Clara's four export-adjustment branches are applied to future years;
- imports are unchanged;
- years 2000, 2005, 2010, 2015 and 2020 are unchanged;
- excluded products are unchanged;
- 2020 `0/0` plus zero future trade is treated as safe `NO_TRADE` with factor `1`;
- undefined or unsafe cases with future trade are blockers;
- the `alpha = -1`, future exports `= 0` case is handled before division, so it correctly returns factor `1`.

The production backend runs in strict mode by default: if any blocker exists, the iteration is not released for injection.

In [ ]:
def demo_row(country, pathway, product, year, imports, exports):
    return {
        "Country": country,
        "CountryID": country,
        "Pathway": pathway,
        "Product": product,
        "Year": year,
        "Import_quantity": imports,
        "Export_quantity": exports,
    }

# Small deterministic iteration example.
iteration_rows = [
    # Canonical case normalization: Barley + barley belong to one global product.
    demo_row("SWE", "CurrentTrends", "Barley", 2020, 0, 100),
    demo_row("ARG", "CurrentTrends", "barley", 2020, 100, 100),
    demo_row("SWE", "CurrentTrends", "Barley", 2030, 0, 150),
    demo_row("ARG", "CurrentTrends", "barley", 2030, 100, 150),

    # alpha = -1 zero-export safety case.
    demo_row("SWE", "CurrentTrends", "lentil", 2020, 100, 0),
    demo_row("SWE", "CurrentTrends", "lentil", 2030, 50, 0),

    # True no-trade product.
    demo_row("ARG", "CurrentTrends", "test_zero", 2020, 0, 0),
    demo_row("ARG", "CurrentTrends", "test_zero", 2030, 0, 0),
]

result = calculate_trade_adjustment(iteration_rows)

print("Adjusted country rows:", len(result["adjusted_rows"]))
print("Global QA rows:", len(result["global_qa"]))
print("Blockers:", len(result["blockers"]))

In [ ]:
print("Global QA")
for row in result["global_qa"]:
    pprint(row)

## 3. Injection response contract

The backend should return a minimal country payload after an iteration is successfully adjusted.

The response uses:

- `Pathway`
- canonical `ProductKey`
- `Year`
- `ImportsAdj`
- `ExportsAdj`

The Office.js add-in should match these against `FullDataTradeAdj` using:

`Pathway + Product.strip().casefold() + Year`

The add-in should preserve the workbook's native product labels; the canonical key is for matching only.

In [ ]:
arg_injection = build_injection_payload(
    result["adjusted_rows"],
    country_id="ARG",
)

pprint(arg_injection)

## 4. Blocking behavior

A product with 2020 global imports = exports = 0 cannot define `alpha`.

- If all future trade is also zero → safe `NO_TRADE`.
- If future trade becomes non-zero → **block the iteration for manual review**.

This is intentional. The web application should show the blocker clearly rather than silently producing `NaN`, `Inf`, or an arbitrary adjustment.

In [ ]:
unsafe_rows = [
    demo_row("A", "CurrentTrends", "new_product", 2020, 0, 0),
    demo_row("A", "CurrentTrends", "new_product", 2030, 10, 0),
]

try:
    calculate_trade_adjustment(unsafe_rows)
except TradeAdjustmentBlocked as exc:
    print(str(exc))
    pprint(exc.blockers)

## 5. Run the production unit tests

The GitHub package includes `tests/test_trade_engine.py`.

These tests cover:

- minimal add-in submission validation and stable report hashing;
- `Barley` / `barley` canonical merging;
- historical rows unchanged;
- future export adjustment;
- excluded products unchanged;
- true zero-trade handling;
- undefined 2020 baseline with future trade blocking;
- the `alpha = -1`, zero-export edge case;
- generation of an injection payload across available pathways.

In [ ]:
import subprocess
import sys

completed = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_trade_engine.py"],
    text=True,
    capture_output=True,
)

print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError("Unit tests failed.")

## 6. What belongs in the web API layer

Keep these responsibilities **outside** `trade_engine.py`:

- authentication / username identity;
- database storage;
- `run_id` generation;
- iteration creation and closing;
- submission timestamps;
- deciding when all required country/pathway reports are present;
- API endpoints;
- returning adjusted results to the Excel add-in.

The engine should remain a pure calculation module. This makes it easy to test and prevents Excel-specific or server-specific code from contaminating the validated methodology.

### Recommended backend flow

> The routes below describe the backend flow conceptually. The FastAPI application code is the source of truth for exact production routes.

1. `POST /submissions`
   - validate with `normalize_submission`;
   - store the minimal rows;
   - store `report_hash`;
   - return server-generated `run_id`.

2. Admin/system closes an iteration.

3. Load all stored rows for that iteration and call:
   - `calculate_trade_adjustment(all_rows, strict=True)`.

4. Store the adjusted rows and QA.

5. `GET /adjustments/{run_id}` or equivalent:
   - identify the participant/country;
   - call `build_injection_payload`;
   - return the minimal adjusted trade JSON.

The Excel add-in then performs the already-validated workbook-side injection and verification.